In [1]:
import os, sys

from chapter7.prompt import format_input

sys.path.append(os.pardir)

In [2]:
from chapter5.gpt_download import download_and_load_gpt2
from previous_chapters import GPTModel
from previous_chapters import load_weights_into_gpt

In [3]:
BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True,
}

In [4]:
model_configs = {
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16}
}

In [5]:
CHOOSE_MODEL = "gpt2-medium (355M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

In [6]:
model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")

In [7]:
settings, params = download_and_load_gpt2(
    model_size = model_size,
    models_dir="gpt2"
)

checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 90.2kiB/s]
encoder.json: 100%|██████████| 1.04M/1.04M [00:01<00:00, 971kiB/s] 
hparams.json: 100%|██████████| 91.0/91.0 [00:00<00:00, 65.4kiB/s]
model.ckpt.data-00000-of-00001: 100%|██████████| 1.42G/1.42G [27:07<00:00, 872kiB/s]  
model.ckpt.index: 100%|██████████| 10.4k/10.4k [00:00<00:00, 6.58MiB/s]
model.ckpt.meta: 100%|██████████| 927k/927k [00:01<00:00, 770kiB/s] 
vocab.bpe: 100%|██████████| 456k/456k [00:00<00:00, 706kiB/s] 


In [8]:
model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

<bound method Module.eval of GPTModel(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_resid): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
 

In [30]:
import torch
from data_loader import val_data
from prompt import format_input

In [31]:
torch.manual_seed(123)
input_text = format_input(val_data[0])
print(input_text)

Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
Rewrite the sentence using a simile.

### Input:
The car is very fast.


In [32]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [35]:
from previous_chapters import generate, text_to_token_ids, token_ids_to_text
token_ids = generate(
    model=model,
    idx = text_to_token_ids(input_text, tokenizer=tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256
)

In [36]:
generate_text = token_ids_to_text(token_ids, tokenizer=tokenizer)
print(generate_text[len(input_text):].strip())

### Output:

The car is very slow.

### Instruction:

Write a response that appropriately completes the request.

### Input:
